# CGR-MAT v1.5 — Adaptive Runtime + Linkage-Fixed Pipeline

This notebook does **not crash when a GPU is unavailable**. It automatically chooses:

- **full** scientific mode when CUDA/T4 is available;
- **smoke** structural-validation mode on CPU, clearly labelled as non-scientific.

It preserves the verified linkage cohort: 463 confirmed labelled appendicitis patients, 445 raw-ultrasound-linked patients, and 18 labelled cases without usable `US_Number`. All artifacts are saved to Google Drive.


In [ ]:
!pip -q install catboost==1.2.8 openpyxl==3.1.5 requests


In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT = Path('/content/drive')
DRIVE_ROOT = MOUNT / 'MyDrive'
if not DRIVE_ROOT.exists():
    try:
        drive.mount(str(MOUNT), force_remount=False, timeout_ms=120000)
    except Exception as first_error:
        print('First Drive mount attempt failed:', first_error)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount(str(MOUNT), force_remount=True, timeout_ms=120000)
if not DRIVE_ROOT.exists():
    raise RuntimeError('Google Drive is required so checkpoints are not lost. Enable pop-ups/cookies and rerun this cell.')
print('✓ Google Drive is ready:', DRIVE_ROOT)


In [ ]:
import hashlib
import pandas as pd
import requests
from IPython.display import display

CACHE = DRIVE_ROOT / 'MAT-Appendix' / 'data_cache' / 'regensburg'
CACHE.mkdir(parents=True, exist_ok=True)
XLSX = CACHE / 'app_data.xlsx'
URL = 'https://zenodo.org/records/7711412/files/app_data.xlsx?download=1'
EXPECTED_MD5 = 'd17a803f5e27532e518676a38f588b59'

def md5(path):
    h = hashlib.md5()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

if not XLSX.exists() or md5(XLSX) != EXPECTED_MD5:
    print('Downloading official app_data.xlsx for preflight...')
    response = requests.get(URL, timeout=180)
    response.raise_for_status()
    XLSX.write_bytes(response.content)
if md5(XLSX) != EXPECTED_MD5:
    raise RuntimeError('Official app_data.xlsx checksum mismatch.')

df = pd.read_excel(XLSX, engine='openpyxl')
norm = lambda s: s.fillna('').astype(str).str.strip().str.lower().str.replace('_', ' ', regex=False)
diagnosis = norm(df['Diagnosis'])
severity = norm(df['Severity'])
confirmed = diagnosis.eq('appendicitis') & severity.isin({'complicated', 'uncomplicated'})
audit = df.loc[confirmed, ['US_Number', 'Diagnosis', 'Severity']].copy()
audit['target'] = severity.loc[audit.index].map({'uncomplicated': 0, 'complicated': 1}).astype(int)
audit['us_number_numeric'] = pd.to_numeric(audit['US_Number'], errors='coerce')
linked = audit.loc[audit['us_number_numeric'].notna()]
unlinked = audit.loc[audit['us_number_numeric'].isna()]
table = pd.DataFrame([
    {'group': 'all_confirmed_appendicitis_with_observed_severity', 'patients': len(audit), 'complicated': int(audit.target.sum()), 'uncomplicated': int((audit.target == 0).sum())},
    {'group': 'multimodal_linked_confirmed_appendicitis', 'patients': len(linked), 'complicated': int(linked.target.sum()), 'uncomplicated': int((linked.target == 0).sum())},
    {'group': 'confirmed_appendicitis_without_us_number', 'patients': len(unlinked), 'complicated': int(unlinked.target.sum()), 'uncomplicated': int((unlinked.target == 0).sum())},
])
print('SCHEMA PREFLIGHT — MUST PASS BEFORE TRAINING')
display(table)
expected = [(463, 118, 345), (445, 116, 329), (18, 2, 16)]
actual = [tuple(map(int, row)) for row in table[['patients', 'complicated', 'uncomplicated']].to_numpy()]
if actual != expected:
    raise RuntimeError(f'Schema preflight failed. Expected {expected}, found {actual}.')
print('✓ Schema preflight passed.')


In [ ]:
import json
import os
import torch
from datetime import datetime, timezone

HAS_GPU = torch.cuda.is_available()
REQUESTED_MODE = os.environ.get('CGR_MAT_REQUESTED_MODE', 'auto').strip().lower()
if REQUESTED_MODE not in {'auto', 'full', 'quick', 'smoke'}:
    raise ValueError('CGR_MAT_REQUESTED_MODE must be auto, full, quick, or smoke.')

if REQUESTED_MODE == 'auto':
    SELECTED_MODE = 'full' if HAS_GPU else 'smoke'
elif REQUESTED_MODE == 'full' and not HAS_GPU:
    print('⚠ Full mode requested but no CUDA GPU is attached. Falling back to smoke mode instead of crashing.')
    SELECTED_MODE = 'smoke'
else:
    SELECTED_MODE = REQUESTED_MODE

os.environ['CGR_MAT_RUN_MODE'] = SELECTED_MODE
os.environ['CGR_MAT_USE_DRIVE'] = '1'
os.environ['CGR_MAT_FORCE_RESTART'] = '0'
os.environ['CGR_MAT_PRETRAINED'] = '1' if HAS_GPU else '0'

decision = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'cuda_available': HAS_GPU,
    'gpu_name': torch.cuda.get_device_name(0) if HAS_GPU else None,
    'requested_mode': REQUESTED_MODE,
    'selected_mode': SELECTED_MODE,
    'scientific_result': bool(HAS_GPU and SELECTED_MODE == 'full'),
}
runtime_file = DRIVE_ROOT / 'MAT-Appendix' / 'runtime_decision_v1_5.json'
runtime_file.parent.mkdir(parents=True, exist_ok=True)
runtime_file.write_text(json.dumps(decision, indent=2), encoding='utf-8')
print(json.dumps(decision, indent=2))
if SELECTED_MODE == 'smoke':
    print('⚠ CPU smoke mode validates the complete pipeline and artifact exports, but its metrics are NOT paper results.')
else:
    print('✓ Scientific GPU run selected.')


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = 'a072de95190d214177bbf3091cf98ab982e9ce5e'
LOADER_URL = (
    'https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/'
    f'{SOURCE_COMMIT}/src/cgr_mat/cgr_mat_verified_loader_v1_4.py'
)
print('Loading pinned linkage-fixed CGR-MAT loader...')
print('Source commit:', SOURCE_COMMIT)
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
print('Loader SHA256:', hashlib.sha256(loader_bytes).hexdigest())
loader = loader_bytes.decode('utf-8')
exec(compile(loader, LOADER_URL, 'exec'), globals(), globals())


## Interpretation

A **full GPU run** is the scientific experiment. A **CPU smoke run** only proves that data loading, model construction, training loops, saving, tables and figures execute without a GPU; do not report smoke metrics in a paper.

All outputs are stored under `MyDrive/MAT-Appendix/cgr_mat_runs/`. Reopen the notebook later with a GPU and use Run all; the source-aware run directory will be different from the CPU smoke directory.
